In [1]:
import pickle
import os
import pandas as pd
from string import punctuation
import random

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.tag import pos_tag
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [2]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
eng_stopwords = stopwords.words('english')

### **Preprocessing**

In [3]:
def AlterTag(tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    return 'n'
    


def Preprocessing(docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in eng_stopwords]
    tokens = [stemmer.stem(tok) for tok in tokens]

    tagged = pos_tag(tokens)

    tokens = [lemmatizer.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]

    return tokens


### **Training**

In [4]:

def Training():
    data = pd.read_csv('./financial_dataset.csv')
    X = data['Statement']
    Y = data['Sentiment']

    # Feature
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocessing(text)
        ft = {word: True for word in clean}
        feats.append((ft, label))
    
    random.shuffle(feats)
    
    # Training
    print('Start Training...')
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)
    print('')

    # Info
    print('Model Trained')
    print(f'Accuracy: {acc}')
    print('')

    print('Top 5 Most Informative Features')
    model.show_most_informative_features(5)
    print('')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')
    print('')

    return model

def Load():
    model = None

    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        print('No Model Detected')
        model = Training()
        return model
    

### **Support Function**

In [5]:
def Write_State():
    while True:
        docx = input('Please enter your Statement')
        if len(docx.split()) < 2:
            print('Please Input at least 2 words')
        else:
            return docx

def Analyze_State(docx: str, model):
    if len(docx.split()) < 2:
        print('Please Input the Statement first!!!!')
        return None

    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in eng_stopwords]

    # POS Tag
    print('POS Tag:')
    tagged = pos_tag(tokens)

    for tok, tag in tagged:
        print(f'- {tok} -> {tag}')
    print('')

    # Synoanto
    print('Synonyms & Antonyms')
    for word in tokens:
        print(f'Word: {word}')
        print('=' * (5 + len(word)))

        synsetx = wordnet.synsets(word) 
        synonyms = []
        antonyms = []

        for sys in synsetx:
            for lemma in sys.lemmas():
                synonyms.append(lemma.name())
                for anton in lemma.antonyms():
                    antonyms.append(anton.name())
        
        print('Synonyms:')
        for w in synonyms:
            print(f'(+) {w}')
        print('')

        print('Antonyms:')
        for w in antonyms:
            print(f'(-) {w}')
        print('')

    # Pred
    clean = Preprocessing(docx)
    feats = {word: True for word in clean}
    
    pred = model.classify(feats)
    print(f'The Statement is classified as: {pred}')
    print('')

### **Main**

In [6]:
def Menu():
    model = Load()
    docx = ''

    while True:
        print('1. Write your Statement')
        print('2. Analyze your Statement')
        print('3. End Session')
        cc = input('>> ')
        print('')

        if cc == '1':
            docx = Write_State()
        elif cc == '2':
            Analyze_State(docx, model)
        elif cc == '3':
            print('Alright, Thank You for using my App')
            break
        else:
            print('Invalid Input')
        print('')
        


In [8]:
Menu()

No Model Detected
Start Training...

Model Trained
Accuracy: 0.7753222836095764

Top 5 Most Informative Features
Most Informative Features
                 decreas = True           negati : positi =     29.2 : 1.0
                    drop = True           negati : positi =     24.2 : 1.0
                    fell = True           negati : positi =     21.0 : 1.0
                   staff = True           negati : positi =     16.8 : 1.0
                     lay = True           negati : positi =     15.4 : 1.0

Model Saved

1. Write your Statement
2. Analyze your Statement
3. End Session

Alright, Thank You for using my App
